# Balanceamento dos dados + divisão em treino/teste (70/30)

Neste notebook, faremos:

* Divisão dos dados em treino/teste (70/30); 
* Salvamento de dois datasets separados para reprodutibilidade;
* Balanceamento (Oversampling por duplicação para SINE/LINE + undersampling para LTR) do conjunto de treino.

In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv('../data/TE_dataset_final_pre_processed.csv')

### Divisão treino/teste (70/30)

In [2]:
import os
from sklearn.model_selection import train_test_split


print(f"Dataset original com: {len(df):,} amostras")
print(f"Features no dataset: {', '.join(df.columns)}")
print("-" * 40)

# 1. Separar features (X) e alvo (y)
# X = todas as colunas, EXCETO 'Classe'
X = df.drop(columns=['Classe']) 
# y = apenas a coluna 'Classe'
y = df['Classe']

# 2. Fazer a divisão 70/30 estratificada
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3, 
                                                    stratify=y, 
                                                    random_state=42)
print(f"Divisão concluída:")
print(f"• Amostras de Treino (70%): {len(X_train):,}")
print(f"• Amostras de Teste (30%):  {len(X_test):,}")

# 3. Recombinar os dataframes para salvar
# Nós juntamos as features de treino (X_train) com o alvo de treino (y_train)
df_train = X_train.join(y_train)
df_test = X_test.join(y_test)

# 4. Definir caminhos e salvar os arquivos
output_dir = '../data/train_test_split'
train_path = os.path.join(output_dir, 'train_dataset.csv')
test_path = os.path.join(output_dir, 'test_dataset.csv')

# Garantir que o diretório de saída exista
os.makedirs(output_dir, exist_ok=True) 

# Salvar os arquivos CSV
df_train.to_csv(train_path, index=False)
print(f"\n✅ Dataset de TREINO salvo em: {train_path}")

df_test.to_csv(test_path, index=False)
print(f"✅ Dataset de TESTE salvo em: {test_path}")

print("-" * 40)
print("\nVerificação das colunas no dataset de treino (df_train):")
print(df_train.head())


Dataset original com: 332,436 amostras
Features no dataset: Cromossomo, Sequência de TE, Classe, Comprimento_Log
----------------------------------------
Divisão concluída:
• Amostras de Treino (70%): 232,705
• Amostras de Teste (30%):  99,731

✅ Dataset de TREINO salvo em: ../data/train_test_split\train_dataset.csv
✅ Dataset de TESTE salvo em: ../data/train_test_split\test_dataset.csv
----------------------------------------

Verificação das colunas no dataset de treino (df_train):
       Cromossomo                                    Sequência de TE  \
38822           3  TTGCGGAAGACCCATTTGGTTCCTACAACATTTTGGTTAGGACGTG...   
318634          1  CTGATCTCGTCGCGTCGGGTGCCCTGACTGCGGACGTTGCTGCCAT...   
41419           1  GTGATTATGTTCGCGCCCGGGTCTTGGTGCCCCGAAACACGGGTGT...   
235883         10  TACTGGTGGACCAGAATGAAGAGGGAAATAGCCCAGTATGTATCAG...   
155119          6  GGAGAGTGTTCGACGGGCAGTGAACGGCAAAATCGTCAGTGAGTGT...   

        Comprimento_Log Classe  
38822          9.336885    LTR  
318634       

### Balanceamento conservador (apenas no dataset de treinamento)

Como lidamos com dados genômicos, usaremos uma abordagem "conservadora" (Random Oversampling) que é frequentemente preferida no contexto da bioinformática. Isso porque o risco de overfitting (memorização) do método de duplicar amostras é muitas vezes um preço menor a pagar do que o risco de aprender biologia falsa com o SMOTE, já que esta metodologia pode criar amostras "biologicamente impossíveis". 

In [3]:
def balanceamento_conservador(df):
    """
    Versão corrigida que usa os números REAIS do dataset
    """
    # Primeiro, descobrir os números REAIS
    distribuição_real = df['Classe'].value_counts()
    print("Distribuição REAL do dataset:")
    for classe, count in distribuição_real.items():
        print(f"  {classe}: {count}")
    
    # Definir targets baseado nos números REAIS
    targets = {
        'LTR': 60000,                    # Undersampling
        'TIR': distribuição_real['TIR'],   # Manter REAL 
        'Helitron': distribuição_real['Helitron'], # Manter REAL
        'MITE': distribuição_real['MITE'],         # Manter REAL
        'LINE': 20000,                    # Oversampling moderado
        'SINE': 8000                      # Oversampling moderado
    }
    
    balanced_dfs = []
    
    for classe, target in targets.items():
        class_data = df[df['Classe'] == classe]
        current_size = len(class_data)
        
        print(f"\nProcessando {classe}:")
        print(f"  Atual: {current_size}, Target: {target}")
        
        if current_size > target:
            # UNDERSAMPLING (apenas para LTR)
            sampled = class_data.sample(n=target, random_state=42)
            print(f"  → Undersampling: {current_size} → {len(sampled)}")
        elif current_size < target:
            # OVERSAMPLING (apenas para LINE e SINE)
            n_copias = (target // current_size) + 1
            copies = [class_data] * n_copias
            sampled = pd.concat(copies, ignore_index=True)
            sampled = sampled.sample(n=target, random_state=42)
            print(f"  → Oversampling: {current_size} → {len(sampled)}")
        else:
            # MANTER ORIGINAL (TIR, Helitron, MITE)
            sampled = class_data
            print(f"  → Mantido original: {current_size}")
        
        balanced_dfs.append(sampled)
    
    df_balanced = pd.concat(balanced_dfs, ignore_index=True)
    return df_balanced.sample(frac=1, random_state=42)

# Aplicar
df_balanced_seguro = balanceamento_conservador(df_train)

# Salvar o dataset balanceado
balanced_path = os.path.join(output_dir, 'train_dataset_balanced.csv')
df_balanced_seguro.to_csv(balanced_path, index=False)
print(f"\n✅ Dataset de TREINO BALANCEADO salvo em: {balanced_path}")



Distribuição REAL do dataset:
  LTR: 131077
  TIR: 56717
  MITE: 18036
  Helitron: 16821
  LINE: 7556
  SINE: 2498

Processando LTR:
  Atual: 131077, Target: 60000
  → Undersampling: 131077 → 60000

Processando TIR:
  Atual: 56717, Target: 56717
  → Mantido original: 56717

Processando Helitron:
  Atual: 16821, Target: 16821
  → Mantido original: 16821

Processando MITE:
  Atual: 18036, Target: 18036
  → Mantido original: 18036

Processando LINE:
  Atual: 7556, Target: 20000
  → Oversampling: 7556 → 20000

Processando SINE:
  Atual: 2498, Target: 8000
  → Oversampling: 2498 → 8000

✅ Dataset de TREINO BALANCEADO salvo em: ../data/train_test_split\train_dataset_balanced.csv


In [4]:
# Distribuição final para verificação
print("\nDistribuição FINAL do dataset balanceado:")
print(df_balanced_seguro['Classe'].value_counts())


Distribuição FINAL do dataset balanceado:
Classe
LTR         60000
TIR         56717
LINE        20000
MITE        18036
Helitron    16821
SINE         8000
Name: count, dtype: int64


In [ ]:
display(df_balanced_seguro.head(1000))